# Avaliação final — comparação GPT-4o-mini vs Opus

Junta as avaliações feitas pelo GPT e pelo Opus, calcula notas, percentual de acerto e isola as divergências.

In [5]:
import pandas as pd

path_eval_model = ''

## 1. Carrega as duas avaliações

In [6]:
df_gpt = pd.read_excel(f'bndes_avaliado_gpt.xlsx')
df_opus = pd.read_excel(f'bndes_avaliado_opus.xlsx')

print('GPT  shape :', df_gpt.shape)
print('Opus shape :', df_opus.shape)
print('\nColunas GPT :', df_gpt.columns.tolist())
print('Colunas Opus:', df_opus.columns.tolist())

GPT  shape : (50, 12)
Opus shape : (50, 12)

Colunas GPT : ['id', 'categoria', 'pergunta', 'resposta', 'input_tokens', 'output_tokens', 'total_tokens', 'custo_usd', 'latencia_s', 'erro', 'justificativa_gpt', 'avaliacao_gpt']
Colunas Opus: ['id', 'categoria', 'pergunta', 'resposta', 'input_tokens', 'output_tokens', 'total_tokens', 'custo_usd', 'latencia_s', 'erro', 'justificativa_opus', 'avaliacao_opus']


## 2. Merge das duas avaliações

Usa o `df_opus` como base (que já tem todas as colunas originais + `justificativa_opus`/`avaliacao_opus`) e traz apenas as colunas de avaliação do GPT, ligando por `id`.

In [7]:
df = df_opus.merge(
    df_gpt[['id', 'justificativa_gpt', 'avaliacao_gpt']],
    on='id',
    how='left',
    validate='one_to_one',
)

# Coluna de concordância: True se as duas avaliações deram a mesma nota
df['concordancia'] = df['avaliacao_opus'] == df['avaliacao_gpt']

print('Shape final:', df.shape)
df.head(3)

Shape final: (50, 15)


,id,categoria,pergunta,resposta,input_tokens,output_tokens,total_tokens,custo_usd,latencia_s,erro,justificativa_opus,avaliacao_opus,justificativa_gpt,avaliacao_gpt,concordancia
0,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...,O período de inscrição para o concurso do BNDE...,8490,135,8625,0.001124,12.80,NaN,"Resposta correta. O edital, no subitem 6.2.1, ...",1.0,Correta. O item 6.2.1 fixa o período de inscri...,1.0,True
1,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...,O valor da taxa de inscrição para o concurso d...,6514,67,6581,0.000902,2.62,NaN,Resposta correta. O subitem 6.2.3 estabelece e...,1.0,Correta. O item 6.2.3 estabelece expressamente...,1.0,True
2,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...,O candidato deve realizar a inscrição para o c...,6045,110,6155,0.000858,6.26,NaN,Resposta correta. Os subitens 6.2.1 e 6.3.1 in...,1.0,Correta. Os itens 6.2.1 e 6.3.1 indicam inscri...,1.0,True


## 3. Notas totais e percentual de acerto

In [9]:
n_perguntas = len(df)

nota_gpt = df['avaliacao_gpt'].sum()
nota_opus = df['avaliacao_opus'].sum()

pct_gpt = nota_gpt / n_perguntas
pct_opus = nota_opus / n_perguntas

resumo = pd.DataFrame({
    'avaliador':       ['ChatGPT', 'Opus'],
    'pontos':          [nota_gpt, nota_opus],
    'total_perguntas': [n_perguntas, n_perguntas],
    'pct_acerto':      [pct_gpt, pct_opus],
})
resumo['pct_acerto'] = resumo['pct_acerto'].map('{:.1%}'.format)
resumo

,avaliador,pontos,total_perguntas,pct_acerto
0,ChatGPT,44.5,50,89.0%
1,Opus,45.5,50,91.0%


### Distribuição das notas por avaliador

In [11]:
dist = pd.DataFrame({
    'ChatGPT': df['avaliacao_gpt'].value_counts().sort_index(),
    'Opus':        df['avaliacao_opus'].value_counts().sort_index(),
}).fillna(0).astype(int)
dist.index.name = 'nota'
dist

,ChatGPT,Opus
nota,,
0.0,2,3
0.5,7,3
1.0,41,44


## 4. Concordância

In [12]:
n_concord = df['concordancia'].sum()
n_diverg = (~df['concordancia']).sum()

print(f'Concordâncias : {n_concord} / {n_perguntas} ({n_concord / n_perguntas:.1%})')
print(f'Divergências  : {n_diverg} / {n_perguntas} ({n_diverg / n_perguntas:.1%})')

Concordâncias : 44 / 50 (88.0%)
Divergências  : 6 / 50 (12.0%)


## 5. DataFrame das divergências

Apenas as linhas em que GPT e Opus discordaram, com as duas justificativas lado a lado para inspeção.

In [13]:
df_divergencias = df.loc[~df['concordancia'], [
    'id',
    'categoria',
    'pergunta',
    'resposta',
    'avaliacao_gpt',
    'justificativa_gpt',
    'avaliacao_opus',
    'justificativa_opus',
]].copy()

df_divergencias['delta'] = df_divergencias['avaliacao_opus'] - df_divergencias['avaliacao_gpt']

print(f'{len(df_divergencias)} divergência(s) encontrada(s)\n')
df_divergencias

6 divergência(s) encontrada(s)



,id,categoria,pergunta,resposta,avaliacao_gpt,justificativa_gpt,avaliacao_opus,justificativa_opus,delta
12,13,Dados sobre o Concurso,Qual é o prazo de validade do concurso BNDES?,O prazo de validade do concurso do BNDES será ...,0.5,Meio certa. O item 1.7 fixa validade de 2 anos...,1.0,Resposta correta. O subitem 1.7 estabelece que...,0.5
16,17,Dados sobre o Concurso,Existe possibilidade de impugnar o edital do c...,"Sim, é possível impugnar o edital do concurso ...",0.0,"Errada. O edital não prevê, de forma expressa,...",0.5,Resposta meio certa. A pergunta é sobre 'impug...,0.5
19,20,Dados sobre o Cargo,Quantas vagas imediatas existem para Ciência d...,No concurso do BNDES para o cargo de Analista ...,0.5,Meio certa. O Anexo I mostra 11 vagas imediata...,0.0,Resposta errada. O Anexo I separa 'VAGAS' de '...,-0.5
25,26,Dados sobre o Cargo,Qual escolaridade é exigida para concorrer à á...,Para concorrer à área de Ciência de Dados no c...,0.5,"Meio certa. O item 2.1.8.1 exige, para Ciência...",0.0,"Resposta errada. O subitem 2.1.8.1 do edital, ...",-0.5
38,39,Dados sobre a Prova,Existe penalidade por erro na prova objetiva d...,"Sim, existem penalidades relacionadas a erros ...",0.5,Meio certa. Os itens 9.18 e 9.18.1 dizem que m...,1.0,Resposta correta. O subitem 9.18 e 9.18.1 esta...,0.5
43,44,Dados sobre a Prova,Quantas provas discursivas de Ciência de Dados...,No concurso do BNDES para o cargo de Analista ...,0.5,Meio certa. O item 8.1.6 prevê correção de até...,1.0,Resposta correta. O subitem 8.1.6 estabelece q...,0.5


### Inspeção rápida das divergências (texto completo)

In [14]:
for _, row in df_divergencias.iterrows():
    print('=' * 80)
    print(f"ID {row['id']} | {row['categoria']}")
    print(f"P: {row['pergunta']}")
    print(f"\n  GPT  ({row['avaliacao_gpt']}): {row['justificativa_gpt']}")
    print(f"\n  OPUS ({row['avaliacao_opus']}): {row['justificativa_opus']}")
    print()

ID 13 | Dados sobre o Concurso
P: Qual é o prazo de validade do concurso BNDES?

  GPT  (0.5): Meio certa. O item 1.7 fixa validade de 2 anos a contar da publicação do Edital de Homologação dos resultados finais, prorrogável uma única vez por igual período. A resposta acerta essa regra, mas acrescenta conclusão temporal e afirma que o prazo 'se estenderá até a homologação', o que é impreciso, pois a validade começa na homologação.

  OPUS (1.0): Resposta correta. O subitem 1.7 estabelece que o prazo de validade é de 2 (dois) anos a contar da publicação do Edital de Homologação, prorrogável uma única vez por igual período, a critério exclusivo do BNDES.

ID 17 | Dados sobre o Concurso
P: Existe possibilidade de impugnar o edital do concurso BNDES?

  GPT  (0.0): Errada. O edital não prevê, de forma expressa, impugnação do edital. Os itens citados pela resposta tratam de recursos ou contestações específicas, como isenção (6.10), questões/gabaritos (10.1) e outras situações, não de impugn

## 6. (opcional) Salva o consolidado

In [15]:
out_path = f'bndes_avaliado_consolidado.xlsx'
df.to_excel(out_path, index=False)
print(f'Consolidado salvo em: {out_path}')

Consolidado salvo em: bndes_avaliado_consolidado.xlsx
